# Support Vector Machines


In this lab we will explore several datasets with SVMs. The assets folder contains several datasets (in order of complexity):

1. Breast cancer
- Spambase
- Car evaluation
- Mushroom

For each of these a `.names` file is provided with details on the origin of data.

In [353]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif
from sklearn.svm import SVC, LinearSVC
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score, learning_curve
from sklearn.metrics import confusion_matrix, classification_report


# Exercise 1: Breast Cancer



## 1.a: Load the Data
Use `pandas.read_csv` to load the data and assess the following:
- Are there any missing values?
- Are the features categorical or numerical?
- Are the values normalized?
- How many classes are there in the target?

Perform what's necessary to get to a point where you have a feature matrix `X` and a target vector `y`, both with only numerical entries.

In [354]:
# Read in the dataset
data_car = pd.read_csv('car.csv')
data_mushroom = pd.read_csv('mushroom.csv')
data_spambase = pd.read_csv('spambase.csv')
data_breast_cancer = pd.read_csv('breast_cancer.csv')


In [355]:
data_breast_cancer.head()
data_breast_cancer.dtypes





,0
Sample_code_number,int64
Clump_Thickness,int64
Uniformity_of_Cell_Size,int64
Uniformity_of_Cell_Shape,int64
Marginal_Adhesion,int64
Single_Epithelial_Cell_Size,int64
Bare_Nuclei,object
Bland_Chromatin,int64
Normal_Nucleoli,int64
Mitoses,int64


In [356]:
# Convert Bare_Nuclei to numeric and change missing values to NaN
data_breast_cancer['Bare_Nuclei'] = pd.to_numeric(
    data_breast_cancer['Bare_Nuclei'],
    errors='coerce'
)
data_breast_cancer.head()




,Sample_code_number,Clump_Thickness,Uniformity_of_Cell_Size,Uniformity_of_Cell_Shape,Marginal_Adhesion,Single_Epithelial_Cell_Size,Bare_Nuclei,Bland_Chromatin,Normal_Nucleoli,Mitoses,Class
0,1000025,5,1,1,1,2,1.0,3,1,1,2
1,1002945,5,4,4,5,7,10.0,3,2,1,2
2,1015425,3,1,1,1,2,2.0,3,1,1,2
3,1016277,6,8,8,1,3,4.0,3,7,1,2
4,1017023,4,1,1,3,2,1.0,3,1,1,2


In [357]:
# Drop the rows that contain a missing value in Bare_Nuclei
breast_cancer =data_breast_cancer.dropna(subset=['Bare_Nuclei'])
df = pd.DataFrame(breast_cancer)
df.head()

,Sample_code_number,Clump_Thickness,Uniformity_of_Cell_Size,Uniformity_of_Cell_Shape,Marginal_Adhesion,Single_Epithelial_Cell_Size,Bare_Nuclei,Bland_Chromatin,Normal_Nucleoli,Mitoses,Class
0,1000025,5,1,1,1,2,1.0,3,1,1,2
1,1002945,5,4,4,5,7,10.0,3,2,1,2
2,1015425,3,1,1,1,2,2.0,3,1,1,2
3,1016277,6,8,8,1,3,4.0,3,7,1,2
4,1017023,4,1,1,3,2,1.0,3,1,1,2


In [358]:
data_breast_cancer['Class'] = data_breast_cancer['Class'].map({
    2: 0,   # benign
    4: 1    # malignant
})
data_breast_cancer.head()



,Sample_code_number,Clump_Thickness,Uniformity_of_Cell_Size,Uniformity_of_Cell_Shape,Marginal_Adhesion,Single_Epithelial_Cell_Size,Bare_Nuclei,Bland_Chromatin,Normal_Nucleoli,Mitoses,Class
0,1000025,5,1,1,1,2,1.0,3,1,1,0
1,1002945,5,4,4,5,7,10.0,3,2,1,0
2,1015425,3,1,1,1,2,2.0,3,1,1,0
3,1016277,6,8,8,1,3,4.0,3,7,1,0
4,1017023,4,1,1,3,2,1.0,3,1,1,0


## 1.b: Model Building

- What's the baseline for the accuracy?
- Initialize and train a linear svm. What's the average accuracy score with a 3-fold cross validation?
- Repeat using an rbf classifier. Compare the scores. Which one is better?
- Are your features normalized? if not, try normalizing and repeat the test. Does the score improve?
- What's the best model?
- Print a confusion matrix and classification report for your best model using:
        train_test_split(X, y, stratify=y, test_size=0.33, random_state=42)

**Check** to decide which model is best, look at the average cross validation score. Are the scores significantly different from one another?

In [359]:
# Set our feature and target variables

X = data_breast_cancer.drop('Class', axis=1).dropna()
y = data_breast_cancer.loc[X.index, 'Class']


In [360]:
linear_svm = LinearSVC(max_iter=10000, random_state=42)

linear_svm.fit(X, y)

score = linear_svm.score(X, y)
print("Linear SVC accuracy on the training data:", score)


Linear SVC accuracy on the training data: 0.6500732064421669


In [361]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

rbf_svm = SVC(kernel='rbf', random_state=42)

rbf_svm.fit(X_scaled, y)

score = rbf_svm.score(X_scaled, y)
print("RBF SVC accuracy on the training data:", score)

RBF SVC accuracy on the training data: 0.9751098096632503


In [362]:
# Now we normalize our data and repeat our models


scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # X must be clean, without NaN

linear_svm = LinearSVC(penalty='l2', max_iter=10000, random_state=42)
linear_scores = cross_val_score(linear_svm, X_scaled, y, cv=3)
print("Linear SVC (L2) 3-fold CV accuracy:", linear_scores.mean())


rbf_svm = SVC(kernel='rbf', random_state=42)
rbf_scores = cross_val_score(rbf_svm, X_scaled, y, cv=3)
print("RBF SVC 3-fold CV accuracy:", rbf_scores.mean())


Linear SVC (L2) 3-fold CV accuracy: 0.9590514465311589
RBF SVC 3-fold CV accuracy: 0.9663485070974057


In [363]:

linear_svm = LinearSVC(penalty='l2', max_iter=10000, random_state=42)

linear_svm.fit(X_scaled, y)
score = linear_svm.score(X_scaled, y)
print("Linear SVC accuracy:", score)


Linear SVC accuracy: 0.9707174231332357


In [364]:
rbf_svm = SVC(kernel='rbf', random_state=42)

rbf_svm.fit(X_scaled, y)

score = rbf_svm.score(X_scaled, y)
print("RBF SVC accuracy:", score)


RBF SVC accuracy: 0.9751098096632503


In [365]:
# Print a confusion matrix and classification report for our best performing model:

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.33, stratify=y, random_state=42
)

best_model = SVC(kernel='rbf', random_state=42)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Confusion Matrix:
[[141   6]
 [  3  76]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.96      0.97       147
           1       0.93      0.96      0.94        79

    accuracy                           0.96       226
   macro avg       0.95      0.96      0.96       226
weighted avg       0.96      0.96      0.96       226



**Check:** Are there more false positives or false negatives? Is this good or bad?

In [366]:
cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()

print("Confusion Matrix:\n", cm)
print(f"False Positives (FP): {FP}")
print(f"False Negatives (FN): {FN}")


if FP > FN:
    print("Il y a plus de faux positifs que de faux négatifs.")
else:
    print("Il y a plus de faux négatifs que de faux positifs.")

if FN > FP:
    print("⚠️ Plus de faux négatifs : cela peut être dangereux car certaines maladies ne sont pas détectées.")
else:
    print("✅ Plus de faux positifs : moins grave, mais peut provoquer des tests ou stress inutiles.")


Confusion Matrix:
 [[141   6]
 [  3  76]]
False Positives (FP): 6
False Negatives (FN): 3
Il y a plus de faux positifs que de faux négatifs.
✅ Plus de faux positifs : moins grave, mais peut provoquer des tests ou stress inutiles.


## 1.c: Feature Selection

Use any of the strategies offered by `sklearn` to select the most important features.

Repeat the cross validation with only those 5 features. Does the score change?

In [367]:

def select_k_best_features(X, y, k):
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    selector = SelectKBest(score_func=chi2, k=k)
    X_selected = selector.fit_transform(X_scaled, y)
    return X_selected, selector


In [368]:
# Select the 5 best features from X
X_top5, selector = select_k_best_features(X, y, k=5)

selected_features = X.columns[selector.get_support()]

print("Top 5 selected features:")
print(selected_features)


Top 5 selected features:
Index(['Uniformity_of_Cell_Size', 'Uniformity_of_Cell_Shape',
       'Marginal_Adhesion', 'Bare_Nuclei', 'Normal_Nucleoli'],
      dtype='object')


In [369]:
# Repeat the cross validation with only those 5 features:
# Linear:
linear_svm = LinearSVC(penalty='l2', max_iter=10000, random_state=42)
linear_scores_top5 = cross_val_score(linear_svm, X_top5, y, cv=3)
print("Linear SVC (top 5 features) 3-fold CV accuracy:", linear_scores_top5.mean())




Linear SVC (top 5 features) 3-fold CV accuracy: 0.9575894582270655


In [370]:
# RBF
rbf_svm = SVC(kernel='rbf', random_state=42)
rbf_scores_top5 = cross_val_score(rbf_svm, X_top5, y, cv=3)
print("RBF SVC (top 5 features) 3-fold CV accuracy:", rbf_scores_top5.mean())

RBF SVC (top 5 features) 3-fold CV accuracy: 0.9546590411417678


##  1.d: Grid Search

Use the grid_search function to explore different kernels and values for the C parameter.

- Can you improve on your best previous score?
- Print the best parameters and the best score

In [371]:


param_grid = {
    'kernel': ['linear', 'rbf', 'poly'],
    'C': [0.01, 0.1, 1, 10, 100]
}

svc = SVC(random_state=42)

grid_search = GridSearchCV(
    estimator=svc,
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_scaled, y)

print("Best parameters found:", grid_search.best_params_)
print("Best 3-fold CV accuracy:", grid_search.best_score_)


Best parameters found: {'C': 0.01, 'kernel': 'linear'}
Best 3-fold CV accuracy: 0.9692789241827034


# Exercise 2
Now that you've completed steps 1.a through 1.d it's time to tackle some harder datasets. But before we do that, let's encapsulate a few things into functions so that it's easier to repeat the analysis.

## 2.a: Cross Validation
Implement a function `do_cv(model, X, y, cv)` that does the following:
- Calculates the cross validation scores
- Prints the model
- Prints and returns the mean and the standard deviation of the cross validation scores

In [372]:

def do_cv(model, X, y, cv=3):
    scores = cross_val_score(model, X, y, cv=cv)

    print("Model:", model)

    mean_score = np.mean(scores)
    std_score = np.std(scores)

    print(f"Cross-validation scores: {scores}")
    print(f"Mean CV accuracy: {mean_score:.4f}")
    print(f"Standard deviation: {std_score:.4f}")

    return mean_score, std_score


## 2.b: Confusion Matrix and Classification report
Implement a function `do_cm_cr(model, X, y, names)` that automates the following:
- Split the data using `train_test_split(X, y, stratify=y, test_size=0.33, random_state=42)`
- Fit the model
- Prints confusion matrix and classification report in a nice format

**Hint:** names is the list of target classes

In [373]:


def do_cm_cr(model, X, y, names):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.33, stratify=y, random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)
    cm_df = pd.DataFrame(cm, index=names, columns=names)

    print("Confusion Matrix:")
    print(cm_df)
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=names))


## 2.c: Grid Search
Implement a function `do_grid_search(model, parameters)` that automates the grid search by doing:
- Calculate grid search
- Print best parameters
- Print best score
- Return best estimator

In [374]:

def do_grid_search(model, parameters, X, y, cv=3, scoring='accuracy'):

    grid_search = GridSearchCV(
        estimator=model,
        param_grid=parameters,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    grid_search.fit(X, y)

    print("Best parameters found:", grid_search.best_params_)
    print(f"Best {scoring} score: {grid_search.best_score_:.4f}")

    return grid_search.best_estimator_


# Exercise 3
Using the functions above, analyze the Spambase dataset.

Notice that now you have many more features. Focus your attention on step C => feature selection

- Load the data and get to X, y
- Select the 15 best features
- Perform grid search to determine best model

In [375]:
# Load the dataset:
data_spambase = pd.read_csv("spambase.csv")
data_spambase.head()


,word_freq_make,word_freq_address,word_freq_all,word_freq_3d,word_freq_our,word_freq_over,word_freq_remove,word_freq_internet,word_freq_order,word_freq_mail,...,char_freq_;,char_freq_(,char_freq_[,char_freq_!,char_freq_$,char_freq_#,capital_run_length_average,capital_run_length_longest,capital_run_length_total,class
0,0.00,0.64,0.64,0.0,0.32,0.00,0.00,0.00,0.00,0.00,...,0.00,0.000,0.0,0.778,0.000,0.000,3.756,61,278,1
1,0.21,0.28,0.50,0.0,0.14,0.28,0.21,0.07,0.00,0.94,...,0.00,0.132,0.0,0.372,0.180,0.048,5.114,101,1028,1
2,0.06,0.00,0.71,0.0,1.23,0.19,0.19,0.12,0.64,0.25,...,0.01,0.143,0.0,0.276,0.184,0.010,9.821,485,2259,1
3,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.00,0.137,0.0,0.137,0.000,0.000,3.537,40,191,1
4,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.00,0.135,0.0,0.135,0.000,0.000,3.537,40,191,1


In [376]:

X = data_spambase.drop('class', axis=1)
y = data_spambase['class']

# Vérification des dimensions
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)



Shape of X: (4601, 57)
Shape of y: (4601,)


In [377]:

# Adaptation du code avec ta fonction
X_top15, selector = select_k_best_features(X, y, k=15)

# Récupérer les noms des 15 features sélectionnées si X est un DataFrame
top15_features = X.columns[selector.get_support()].tolist()
print("Top 15 selected features:", top15_features)


Top 15 selected features: ['word_freq_our', 'word_freq_remove', 'word_freq_order', 'word_freq_receive', 'word_freq_addresses', 'word_freq_free', 'word_freq_business', 'word_freq_your', 'word_freq_000', 'word_freq_money', 'word_freq_hp', 'word_freq_hpl', 'word_freq_george', 'word_freq_labs', 'char_freq_$']


In [378]:
# Perform grid search to determine best model :


param_grid = {
    'kernel': ['linear', 'rbf'],
    'C': [0.01, 1, 100],
}

svc = SVC(random_state=42)

best_model = do_grid_search(svc, param_grid, X_top15, y)




Best parameters found: {'C': 1, 'kernel': 'rbf'}
Best accuracy score: 0.8948


In [379]:
# Perform cross-validation on the best model
mean_score, std_score = do_cv(best_model, X_top15, y, cv=3)

print(f"Best model CV mean accuracy: {mean_score:.4f}")
print(f"Best model CV standard deviation: {std_score:.4f}")



Model: SVC(C=1, random_state=42)
Cross-validation scores: [0.88591917 0.92046936 0.87801696]
Mean CV accuracy: 0.8948
Standard deviation: 0.0184
Best model CV mean accuracy: 0.8948
Best model CV standard deviation: 0.0184


# Exercise 4
Repeat steps 1.a - 1.d for the car dataset. Notice that now features are categorical, not numerical.
- Find a suitable way to encode them
- How does this change our modeling strategy?

Also notice that the target variable `acceptability` has 4 classes. How do we encode them?


In [380]:
# Read in the dataset car.csv
df_car = pd.read_csv('car.csv')
df_car.head()

,buying,maint,doors,persons,lug_boot,safety,acceptability
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc


In [381]:
# Exploratory analysis

print("\nInfo sur le dataset:")
df_car.info()

print("\nStatistiques pour chaque feature:")
for col in df_car.columns:
    print(f"\nColumn: {col}")
    df_car[col].value_counts()

# Vérifier la distribution de la cible
print("\nDistribution de la cible 'acceptability':")
df_car['acceptability'].value_counts()


Info sur le dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1728 entries, 0 to 1727
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   buying         1728 non-null   object
 1   maint          1728 non-null   object
 2   doors          1728 non-null   object
 3   persons        1728 non-null   object
 4   lug_boot       1728 non-null   object
 5   safety         1728 non-null   object
 6   acceptability  1728 non-null   object
dtypes: object(7)
memory usage: 94.6+ KB

Statistiques pour chaque feature:

Column: buying

Column: maint

Column: doors

Column: persons

Column: lug_boot

Column: safety

Column: acceptability

Distribution de la cible 'acceptability':


,count
acceptability,
unacc,1210
acc,384
good,69
vgood,65


In [382]:
# Set the target and features
X = df_car.drop('acceptability', axis=1)
y = df_car['acceptability']

X.shape
y.shape




(1728,)

In [383]:


# Encode the features (categorical → numeric)
ohe = OneHotEncoder(sparse_output=False)  # <-- attention à sparse_output
X_encoded = ohe.fit_transform(X)

print("Shape of X after OneHotEncoding:", X_encoded.shape)

# Encode the target (classes → integers)
le = LabelEncoder()
y_encoded = le.fit_transform(y)  # unacc=0, acc=1, good=2, vgood=3
print("Classes after encoding:", le.classes_)
print("Sample of encoded y:", y_encoded[:10])


Shape of X after OneHotEncoding: (1728, 21)
Classes after encoding: ['acc' 'good' 'unacc' 'vgood']
Sample of encoded y: [2 2 2 2 2 2 2 2 2 2]


In [384]:
# Do a grid search to determine best modeling parameters
param_grid = {
    'kernel': ['linear', 'rbf'],
    'C': [0.1, 1, 10]
}

svc = SVC(random_state=42, cache_size=500)

best_model_car = do_grid_search(svc, param_grid, X_encoded, y_encoded, cv=2)

print("Best model for Car dataset:", best_model_car)


Best parameters found: {'C': 0.1, 'kernel': 'rbf'}
Best accuracy score: 0.7025
Best model for Car dataset: SVC(C=0.1, cache_size=500, random_state=42)


In [385]:
# Select 10 best features and perform another grid search


selector_car = SelectKBest(score_func=chi2, k=10)
X_top10 = selector_car.fit_transform(X_encoded, y_encoded)

top10_feature_indices = selector_car.get_support(indices=True)
print("Indices of top 10 features:", top10_feature_indices)

param_grid = {
    'kernel': ['linear', 'rbf'],
    'C': [0.1, 1, 10]
}

svc = SVC(random_state=42, cache_size=500)

best_model_car_top10 = do_grid_search(svc, param_grid, X_top10, y_encoded, cv=2)
print("Best model using top 10 features:", best_model_car_top10)


Indices of top 10 features: [ 1  3  5  7 12 13 14 18 19 20]
Best parameters found: {'C': 0.1, 'kernel': 'rbf'}
Best accuracy score: 0.7211
Best model using top 10 features: SVC(C=0.1, cache_size=500, random_state=42)


In [386]:
# Select 5 best features and perform another grid search


selector_car5 = SelectKBest(score_func=chi2, k=5)
X_top5 = selector_car5.fit_transform(X_encoded, y_encoded)

top5_feature_indices = selector_car5.get_support(indices=True)
print("Indices of top 5 features:", top5_feature_indices)

param_grid = {
    'kernel': ['linear', 'rbf'],
    'C': [0.1, 1, 10]
}

svc = SVC(random_state=42, cache_size=500)

best_model_car_top5 = do_grid_search(svc, param_grid, X_top5, y_encoded, cv=2)
print("Best model using top 5 features:", best_model_car_top5)


Indices of top 5 features: [ 1 12 13 18 19]
Best parameters found: {'C': 1, 'kernel': 'rbf'}
Best accuracy score: 0.7477
Best model using top 5 features: SVC(C=1, cache_size=500, random_state=42)


In [387]:
# Perform cross-validation on the best model with 5 features
mean_score, std_score = do_cv(best_model_car_top5, X_top5, y_encoded, cv=3)

print(f"Best model (top 5 features) CV mean accuracy: {mean_score:.4f}")
print(f"Best model (top 5 features) CV standard deviation: {std_score:.4f}")



Model: SVC(C=1, cache_size=500, random_state=42)
Cross-validation scores: [0.68576389 0.80381944 0.73958333]
Mean CV accuracy: 0.7431
Standard deviation: 0.0483
Best model (top 5 features) CV mean accuracy: 0.7431
Best model (top 5 features) CV standard deviation: 0.0483



Repeat steps 1.a - 1.d for the mushroom dataset. Notice that now features are categorical, not numerical. This dataset is quite large.
- How does this change our modeling strategy?
- Can we use feature selection to improve this?


In [388]:
df_mushroom = pd.read_csv('mushroom.csv')
df_mushroom.head()


,class,cap-shape,cap-surface,cap-color,bruises?,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g


In [389]:
# Map the class column (e = edible, p = poisonous)

le = LabelEncoder()
df_mushroom['class_encoded'] = le.fit_transform(df_mushroom['class'])

# Vérification
print(df_mushroom[['class', 'class_encoded']].head())
print("Classes mapping:", dict(zip(le.classes_, le.transform(le.classes_))))


  class  class_encoded
0     p              1
1     e              0
2     e              0
3     p              1
4     e              0
Classes mapping: {'e': np.int64(0), 'p': np.int64(1)}


In [390]:
# Look at the data
df_mushroom.head()



,class,cap-shape,cap-surface,cap-color,bruises?,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat,class_encoded
0,p,x,s,n,t,p,f,c,n,k,...,w,w,p,w,o,p,k,s,u,1
1,e,x,s,y,t,a,f,c,b,k,...,w,w,p,w,o,p,n,n,g,0
2,e,b,s,w,t,l,f,c,b,n,...,w,w,p,w,o,p,n,n,m,0
3,p,x,y,w,t,p,f,c,n,n,...,w,w,p,w,o,p,k,s,u,1
4,e,x,s,g,f,n,f,w,b,k,...,w,w,p,w,o,e,n,a,g,0


In [391]:
# Create a map and apply it for each column to convert from letter values to numbers
feature_maps = {}

for col in df_mushroom.columns:
    if col != 'class_encoded':
        unique_vals = df_mushroom[col].unique()
        mapping = {val: idx for idx, val in enumerate(unique_vals)}
        feature_maps[col] = mapping
        df_mushroom[col] = df_mushroom[col].map(mapping)

df_mushroom.head()




,class,cap-shape,cap-surface,cap-color,bruises?,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat,class_encoded
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,1,0,0,1,0,1,0,0,1,0,...,0,0,0,0,0,0,1,1,1,0
2,1,1,0,2,0,2,0,0,1,1,...,0,0,0,0,0,0,1,1,2,0
3,0,0,1,2,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,1
4,1,0,0,3,1,3,0,1,1,0,...,0,0,0,0,0,1,1,2,1,0


In [392]:
# Set the target and features
X = df_mushroom.drop('class_encoded', axis=1)
y = df_mushroom['class_encoded']

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)



Shape of X: (8124, 23)
Shape of y: (8124,)


In [393]:
# 1 – Grid Search Mushroom avec bruit et 5 features
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
import numpy as np

X = df_mushroom.drop('class_encoded', axis=1)
y = df_mushroom['class_encoded']

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_encoded = ohe.fit_transform(X)

rng = np.random.default_rng(42)
noise = rng.normal(0, 0.01, X_encoded.shape)
X_noisy = X_encoded + noise

X_noisy = np.clip(X_noisy, 0, None)

selector = SelectKBest(score_func=mutual_info_classif, k=5)
X_top5 = selector.fit_transform(X_noisy, y)

feature_names = ohe.get_feature_names_out()
selected_features = feature_names[selector.get_support()]
print("Top 5 selected features:", selected_features)

X_train, X_test, y_train, y_test = train_test_split(
    X_top5, y, test_size=0.33, stratify=y, random_state=42
)

param_grid = {
    'kernel': ['linear', 'rbf'],
    'C': [0.01, 0.05, 0.1]
}

svc = SVC(random_state=42, cache_size=500)

best_model_mushroom = do_grid_search(svc, param_grid, X_train, y_train, cv=3)

print("Best model for Mushroom dataset (train set, top 5 features):", best_model_mushroom)


Top 5 selected features: ['class_0' 'class_1' 'odor_3' 'odor_4' 'stalk-surface-above-ring_2']
Best parameters found: {'C': 0.01, 'kernel': 'linear'}
Best accuracy score: 1.0000
Best model for Mushroom dataset (train set, top 5 features): SVC(C=0.01, cache_size=500, kernel='linear', random_state=42)


In [394]:
# Create a model using these optimal parameters

best_params = best_model_mushroom.get_params()

final_model_mushroom = SVC(
    kernel=best_params['kernel'],
    C=best_params['C'],
    cache_size=500,
    random_state=42
)

print("Final SVC model with optimal parameters:")
print(final_model_mushroom)



Final SVC model with optimal parameters:
SVC(C=0.01, cache_size=500, kernel='linear', random_state=42)


In [395]:
# Perform a cross validation on this model
mean_score, std_score = do_cv(final_model_mushroom, X_train, y_train, cv=3)

print(f"Final model CV mean accuracy (train set): {mean_score:.4f}")
print(f"Final model CV standard deviation (train set): {std_score:.4f}")



Model: SVC(C=0.01, cache_size=500, kernel='linear', random_state=42)
Cross-validation scores: [1. 1. 1.]
Mean CV accuracy: 1.0000
Standard deviation: 0.0000
Final model CV mean accuracy (train set): 1.0000
Final model CV standard deviation (train set): 0.0000


In [396]:
# Print the confusion matrix and classification report for this model

class_names = ['edible', 'poisonous']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, stratify=y, random_state=42
)

final_model_mushroom.fit(X_train, y_train)

y_pred = final_model_mushroom.predict(X_test)

do_cm_cr(final_model_mushroom, X, y, names=class_names)


Confusion Matrix:
           edible  poisonous
edible       1389          0
poisonous       0       1292

Classification Report:
              precision    recall  f1-score   support

      edible       1.00      1.00      1.00      1389
   poisonous       1.00      1.00      1.00      1292

    accuracy                           1.00      2681
   macro avg       1.00      1.00      1.00      2681
weighted avg       1.00      1.00      1.00      2681



In [397]:
# See what happens when we select only the 10 best features


selector_mushroom = SelectKBest(score_func=chi2, k=10)
X_top10 = selector_mushroom.fit_transform(X, y)

top10_feature_indices = selector_mushroom.get_support(indices=True)
print("Indices of top 10 features:", top10_feature_indices)


param_grid = {
    'kernel': ['linear', 'rbf'],
    'C': [0.1, 1, 10]
}

svc = SVC(random_state=42, cache_size=500)

best_model_mushroom_top10 = do_grid_search(svc, param_grid, X_top10, y, cv=2)
print("Best model using top 10 features:", best_model_mushroom_top10)


Indices of top 10 features: [ 0  4  5  7  9 12 13 14 19 20]
Best parameters found: {'C': 0.1, 'kernel': 'linear'}
Best accuracy score: 1.0000
Best model using top 10 features: SVC(C=0.1, cache_size=500, kernel='linear', random_state=42)


In [398]:
# Do a grid search to determine best modeling parameters


param_grid = {
    'kernel': ['linear', 'rbf'],
    'C': [0.001, 0.1, 1, 10]
}

svc = SVC(random_state=42, cache_size=500)

best_model_mushroom_top10 = do_grid_search(svc, param_grid, X_top10, y, cv=2)

print("Best SVM model for Mushroom dataset (top 10 features):", best_model_mushroom_top10)


Best parameters found: {'C': 0.1, 'kernel': 'linear'}
Best accuracy score: 1.0000
Best SVM model for Mushroom dataset (top 10 features): SVC(C=0.1, cache_size=500, kernel='linear', random_state=42)


In [399]:


best_params = best_model_mushroom_top10.get_params()

final_model_mushroom_top10 = SVC(
    kernel=best_params['kernel'],
    C=best_params['C'],
    cache_size=500,
    random_state=42
)

print("Final SVC model with optimal parameters (top 10 features):")
print(final_model_mushroom_top10)


Final SVC model with optimal parameters (top 10 features):
SVC(C=0.1, cache_size=500, kernel='linear', random_state=42)


In [400]:
# Perform a cross validation on this model
mean_score, std_score = do_cv(final_model_mushroom_top10, X_top10, y, cv=3)

print(f"Final model (top 10 features) CV mean accuracy: {mean_score:.4f}")
print(f"Final model (top 10 features) CV standard deviation: {std_score:.4f}")


Model: SVC(C=0.1, cache_size=500, kernel='linear', random_state=42)
Cross-validation scores: [1. 1. 1.]
Mean CV accuracy: 1.0000
Standard deviation: 0.0000
Final model (top 10 features) CV mean accuracy: 1.0000
Final model (top 10 features) CV standard deviation: 0.0000


In [401]:


class_names = ['edible', 'poisonous']

X_train, X_test, y_train, y_test = train_test_split(
    X_top10, y, test_size=0.33, stratify=y, random_state=42
)

final_model_mushroom_top10.fit(X_train, y_train)

y_pred = final_model_mushroom_top10.predict(X_test)

do_cm_cr(final_model_mushroom_top10, X_top10, y, names=class_names)


Confusion Matrix:
           edible  poisonous
edible       1389          0
poisonous       0       1292

Classification Report:
              precision    recall  f1-score   support

      edible       1.00      1.00      1.00      1389
   poisonous       1.00      1.00      1.00      1292

    accuracy                           1.00      2681
   macro avg       1.00      1.00      1.00      2681
weighted avg       1.00      1.00      1.00      2681

